## 1. Setup e imports

In [84]:
import pandas as pd 
import numpy as np
from scipy import stats
import plotly.express as px
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

NAVY = "#1C3257"
TERRA = "#AA4B37"
SAND = "#F4EFE6"
INK = "#1A1A1A"
PLOTLY_TEMPLATE = "plotly_white"
PLOTLY_FONT = dict(family="Helvetica, Arial, sans-serif", color=INK, size=13)

df = pd.read_csv("./data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')
df['SeniorCitizen'] = df['SeniorCitizen'].map({0:'No', 1:'Yes'}) # opcional, mas legible
print(f"Shape: {df.shape}")


Shape: (7043, 21)


In [85]:
def clasificar_atributos(
    df: pd.DataFrame,
    hints: dict[str, str] | None = None,
) -> pd.DataFrame:
    """Tabla con dtype, tipo conceptual, n_unique, n_missing y ejemplos por columna.

    Hints permite override manual: {'customerID': 'nominal', 'Contract': 'ordinal'}.
    Heuristica cuando no hay hint:
      - object/string  -> nominal
      - numerico con n_unique == 2 -> binario
      - entero con n_unique <= 10  -> revisar (probable nominal/ordinal codificado)
      - resto numerico -> numerico
    """
    hints = hints or {}
    filas = []
    for col in df.columns:
        s = df[col]
        n_unique = int(s.nunique(dropna=True))
        n_missing = int(s.isna().sum())
        dtype = str(s.dtype)

        if col in hints:
            tipo = hints[col]
        elif pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
            tipo = "nominal"
        elif n_unique == 2:
            tipo = "binario"
        elif pd.api.types.is_integer_dtype(s) and n_unique <= 10:
            tipo = "revisar"
        elif pd.api.types.is_numeric_dtype(s):
            tipo = "numerico"
        else:
            tipo = "revisar"

        ejemplos = s.dropna().unique()[:4]
        filas.append(
            {
                "columna": col,
                "dtype_pandas": dtype,
                "tipo_conceptual": tipo,
                "n_unique": n_unique,
                "n_missing": n_missing,
                "ejemplo_valores": list(ejemplos),
            }
        )
    return pd.DataFrame(filas)

In [86]:
HINTS = {
    "customerID": "nominal",
    "SeniorCitizen": "binario",
    "Contract": "ordinal",
    "Churn": "binario",
    "PaymentMethod": "nominal",
    "PaperlessBilling": "binario",
    "InternetService": "nominal",
    "tenure": "numerico"
}
clasificar_atributos(df, hints=HINTS)

,columna,dtype_pandas,tipo_conceptual,n_unique,n_missing,ejemplo_valores
0,customerID,str,nominal,7043,0,"[7590-VHVEG, 5575-GNVDE, 3668-QPYBK, 7795-CFOCW]"
1,gender,str,nominal,2,0,"[Female, Male]"
2,SeniorCitizen,str,binario,2,0,"[No, Yes]"
3,Partner,str,nominal,2,0,"[Yes, No]"
4,Dependents,str,nominal,2,0,"[No, Yes]"
5,tenure,int64,numerico,73,0,"[1, 34, 2, 45]"
6,PhoneService,str,nominal,2,0,"[No, Yes]"
7,MultipleLines,str,nominal,3,0,"[No phone service, No, Yes]"
8,InternetService,str,nominal,3,0,"[DSL, Fiber optic, No]"
9,OnlineSecurity,str,nominal,3,0,"[No, Yes, No internet service]"


## 2. EDA mínimo (5-8 celdas, no 40)

In [87]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   str    
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [88]:
df.describe()

,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7032.000000
mean,32.371149,64.761692,2283.300441
std,24.559481,30.090047,2266.771362
min,0.000000,18.250000,18.800000
25%,9.000000,35.500000,401.450000
50%,29.000000,70.350000,1397.475000
75%,55.000000,89.850000,3794.737500
max,72.000000,118.750000,8684.800000


**Distribución de Churn (sí está desbalanceado — eso importa).**

In [89]:
churn_dist = (
    df["Churn"]
    .value_counts()
    .rename_axis("Churn")
    .reset_index(name="conteo")
)
churn_dist["porcentaje"] = churn_dist["conteo"] / churn_dist["conteo"].sum() * 100

display(churn_dist)

clase_mayoritaria = churn_dist.loc[churn_dist["conteo"].idxmax()]
clase_minoritaria = churn_dist.loc[churn_dist["conteo"].idxmin()]

print(f"Total de clientes: {churn_dist['conteo'].sum():,}")
print(f"Clase mayoritaria: {clase_mayoritaria['Churn']} ({clase_mayoritaria['porcentaje']:.2f}%)")
print(f"Clase minoritaria: {clase_minoritaria['Churn']} ({clase_minoritaria['porcentaje']:.2f}%)")
print("El target esta desbalanceado; por eso conviene usar particion estratificada y revisar metricas como recall, F1 y ROC-AUC.")

fig = px.bar(
    churn_dist,
    x="Churn",
    y="conteo",
    text=churn_dist["porcentaje"].map(lambda p: f"{p:.1f}%"),
    template=PLOTLY_TEMPLATE,
    color="Churn",
    color_discrete_sequence=[NAVY, TERRA],
    title="Distribucion de Churn"
)
fig.update_traces(textposition="outside")
fig.update_layout(
    font=PLOTLY_FONT,
    width=700,
    height=450,
    showlegend=False,
    yaxis_title="Numero de clientes"
)
fig.show()

,Churn,conteo,porcentaje
0,No,5174,73.463013
1,Yes,1869,26.536987


Total de clientes: 7,043
Clase mayoritaria: No (73.46%)
Clase minoritaria: Yes (26.54%)
El target esta desbalanceado; por eso conviene usar particion estratificada y revisar metricas como recall, F1 y ROC-AUC.


**Distribución de TotalCharges después de castearlo a float.**

In [90]:
ic = df["TotalCharges"].dropna().reset_index(drop=True)
n = len(ic)

# Media: suma de valores / n
media_manual = ic.sum() / n

# Mediana: valor central tras ordenar (promedio de los dos centrales si n es par)
ic_ord = ic.sort_values().reset_index(drop=True)
if n % 2 == 1:
    mediana_manual = ic_ord.iloc[n // 2]
else:
    mediana_manual = (ic_ord.iloc[n // 2 - 1] + ic_ord.iloc[n // 2]) / 2

# Moda: valor con mayor frecuencia
conteos = ic.value_counts()
moda_manual = conteos.index[0]

print(f"n clientes: {n:,}")
print(f"Media    manual: {media_manual:>12,.2f}   pandas: {ic.mean():>12,.2f}")
print(f"Mediana  manual: {mediana_manual:>12,.2f}   pandas: {ic.median():>12,.2f}")
print(f"Moda     manual: {moda_manual:>12,.2f}   pandas: {ic.mode().iloc[0]:>12,.2f}")


fig = px.histogram(df, x="TotalCharges", nbins=60, template=PLOTLY_TEMPLATE,
                   color_discrete_sequence=[NAVY])
fig.add_vline(x=ic.mean(),   line_color=TERRA, line_width=2, annotation_text="media")
fig.add_vline(x=ic.median(), line_color=NAVY,  line_width=2, line_dash="dash", annotation_text="mediana")
fig.update_layout(title="Distribucion de TotalCharges",
                  font=PLOTLY_FONT, width=900, height=450)
fig.show()

n clientes: 7,032
Media    manual:     2,283.30   pandas:     2,283.30
Mediana  manual:     1,397.47   pandas:     1,397.47
Moda     manual:        20.20   pandas:        20.20


**Una matriz de correlación de las columnas numéricas.**

In [91]:
corr_cols = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
df_corr = df[corr_cols].copy()
df_corr["SeniorCitizen"] = df_corr["SeniorCitizen"].map({"Yes": 1, "No": 0})
corr = df_corr.corr().round(2)
fig = px.imshow(corr, text_auto=True,
                color_continuous_scale=[(0, SAND), (0.5, "white"), (1, NAVY)],
                zmin=-1, zmax=1, template=PLOTLY_TEMPLATE)
fig.update_layout(title="Correlacion Pearson", font=PLOTLY_FONT, width=700, height=600)
fig.show()

## 3. Preprocessing

**TotalCharges viene como string con espacios en blanco. Castea con
pd.to_numeric(errors='coerce') y maneja los NaN resultantes (son los clientes con tenure = 0,
decisión tuya cómo tratarlos).**

In [92]:
df = df.dropna(subset=["TotalCharges"])

**customerID se elimina (no es atributo predictor).**

In [93]:
df = df.drop("customerID", axis=1)

**One-hot encoding para categóricas.**

In [94]:
categorical_features = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

# Para regresion: y = TotalCharges, entonces TotalCharges no va en X.
numeric_features_reg = ['tenure', 'MonthlyCharges']

preprocessor_reg = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features_reg),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), categorical_features)
    ]
)

# Para clasificacion: y = Churn, aqui TotalCharges si puede ser predictor.
numeric_features_clf = ['tenure', 'MonthlyCharges', 'TotalCharges']

preprocessor_clf = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features_clf),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

## 4. Regresión: predecir TotalCharges
**4.1 Modelo elegido + justificación**


In [95]:
X_reg = df[numeric_features_reg + categorical_features].copy()
y_reg = df["TotalCharges"].copy()

linear_regression_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_reg),
        ("model", LinearRegression())
    ]
)

print(f"X shape: {X_reg.shape}")
print(f"y shape: {y_reg.shape}")

X shape: (7032, 18)
y shape: (7032,)


**4.2 Entrenamiento + validación cruzada**

In [96]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

cv_results = cross_validate(
    linear_regression_pipeline,
    X_reg,
    y_reg,
    cv=cv,
    scoring=scoring,
    return_estimator=True
)

cv_scores = pd.DataFrame({
    "fold": np.arange(1, cv.get_n_splits() + 1),
    "RMSE": -cv_results["test_rmse"],
    "MAE": -cv_results["test_mae"],
    "R2": cv_results["test_r2"]
})

cv_summary = cv_scores[["RMSE", "MAE", "R2"]].agg(["mean", "std"]).T
cv_summary.columns = ["promedio", "desviacion"]

display(cv_scores.round(3))
display(cv_summary.round(3))

,fold,RMSE,MAE,R2
0,1,673.950,541.023,0.912
1,2,676.263,542.889,0.914
2,3,704.967,566.374,0.905
3,4,706.436,575.387,0.901
4,5,696.518,565.661,0.901


,promedio,desviacion
RMSE,691.627,15.570
MAE,558.267,15.389
R2,0.907,0.006


**4.3 Métricas, pesos y discusión**

In [97]:
weight_frames = []

for fold, estimator in enumerate(cv_results["estimator"], start=1):
    feature_names = estimator.named_steps["preprocessor"].get_feature_names_out()
    coefficients = estimator.named_steps["model"].coef_
    weight_frames.append(
        pd.DataFrame({
            "fold": fold,
            "atributo": feature_names,
            "w": coefficients
        })
    )

weights = pd.concat(weight_frames, ignore_index=True)
weight_summary = (
    weights
    .groupby("atributo")["w"]
    .agg(["mean", "std"])
    .reset_index()
)
weight_summary["abs_mean"] = weight_summary["mean"].abs()

top_weights = weight_summary.sort_values("abs_mean", ascending=False).head(10)
display(top_weights.round(3))

top_attribute = top_weights.iloc[0]
print(f"Atributo con mayor peso absoluto promedio: {top_attribute['atributo']} (w promedio = {top_attribute['mean']:.2f})")

,atributo,mean,std,abs_mean
28,num__tenure,1503.368,11.124,1503.368
27,num__MonthlyCharges,1059.207,67.919,1059.207
10,cat__OnlineBackup_Yes,271.509,24.261,271.509
17,cat__PaymentMethod_Mailed check,227.016,15.212,227.016
1,cat__Contract_Two year,-222.454,19.707,222.454
12,cat__OnlineSecurity_Yes,212.708,16.235,212.708
4,cat__DeviceProtection_Yes,198.478,18.423,198.478
25,cat__TechSupport_Yes,195.582,14.982,195.582
21,cat__StreamingMovies_Yes,105.441,26.351,105.441
23,cat__StreamingTV_Yes,93.073,34.437,93.073


Atributo con mayor peso absoluto promedio: num__tenure (w promedio = 1503.37)


**Discusión**
Usé regresión lineal porque fue el modelo que más entendí y el que más pude relacionar con las diapositivas. segun las diapositivas es facil de interpretar y También es el modelo base sobre el que se entienden Ridge y Lasso, pero considero que era mejor aplicar ridge ya que esta sirve para la multicolinealidad lo que ayuda a que el csv de telco tiene muchos datos relacionados entre si.  investigando mas sobre los modelos me explico que esta intenta encontrar los pesos que reducen el error entre la Y real y la predicha. decidí quedarme con regresión lineal simple para interpretar mejor el modelo y porque no tenía que justificar un valor de alpha ni contestar las preguntas que conllevan.
1. ¿Tu modelo predice bien?,¿RMSE de cuánto es aceptable dado el rango de TotalCharges?

   Respuesta : mi modelo me indica que gracias a MAE la prediccion falla por unos 550, El RMSE es mayor, cerca de 692, lo cual me indica que mi modelo predice bien pero me indica que algunos errores grandes pesan mas y segun R2 mi modelo si logra explicar muy bien una parte alta de la variacion, por lo que refuta la idea de escojer este. 

   El rango aproximado de TotalCharges va de 18.8 a 8684.8 lo que descubrimos en la primera actividad, es un intermedio debido a que el rango es en promedio 8600 comparado con los 690 no es mucho pero podria mejorarse aun porque causaria problemas debido a que podria afectar mucho a unos clientes

2. Si usaste Ridge/Lasso, ¿qué alpha ganó? ¿Cómo cambian los pesos vs. la lineal sin regularizar?

   Respuesta :No usé Ridge ni Lasso por lo que no hay un alpha,pero como mencione anteriormente considero que debi haber usado ridge


3. ¿Qué atributo tiene el peso más alto? ¿Tiene sentido de negocio?

   Respuesta : tenure es el que tiene el peso mas grande, debido a como indica la guia que nos proporciono en la primera actividad mientras mas meses lleve mas grande sera totalcharges, por lo que si tiene sentido de negocio


## 5. Clasificación: predecir Churn (mínimo 2 modelos)

In [98]:
X_clf = df[numeric_features_clf + categorical_features].copy()
y_clf = df["Churn"].map({"No": 0, "Yes": 1})

clf_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_classifier_cv(name, pipeline, X, y, cv):
    y_pred = cross_val_predict(pipeline, X, y, cv=cv, method="predict")
    y_proba = cross_val_predict(pipeline, X, y, cv=cv, method="predict_proba")[:, 1]

    cm = pd.DataFrame(
        confusion_matrix(y, y_pred, labels=[0, 1]),
        index=["Real No", "Real Yes"],
        columns=["Pred No", "Pred Yes"]
    )

    report = pd.DataFrame(
        classification_report(
            y,
            y_pred,
            target_names=["No", "Yes"],
            output_dict=True,
            zero_division=0
        )
    ).T.loc[["No", "Yes"], ["precision", "recall", "f1-score", "support"]]

    auc = roc_auc_score(y, y_proba)
    fpr, tpr, thresholds = roc_curve(y, y_proba)

    roc_df = pd.DataFrame({
        "modelo": name,
        "fpr": fpr,
        "tpr": tpr,
        "threshold": thresholds
    })

    return {
        "modelo": name,
        "confusion_matrix": cm,
        "classification_report": report,
        "roc_auc": auc,
        "roc_curve": roc_df
    }

**5.1 Modelo A: regresión logística + entrenamiento**

In [99]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_clf),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
    ]
)

logistic_results = evaluate_classifier_cv(
    "Regresion logistica",
    logistic_pipeline,
    X_clf,
    y_clf,
    clf_cv
)

print(f"ROC-AUC regresion logistica: {logistic_results['roc_auc']:.3f}")
display(logistic_results["confusion_matrix"])
display(logistic_results["classification_report"].round(3))

ROC-AUC: 0.845


,Pred No,Pred Yes
Real No,3761,1402
Real Yes,375,1494


,precision,recall,f1-score,support
No,0.909,0.728,0.809,5163.0
Yes,0.516,0.799,0.627,1869.0


**5.2 Modelo B: árbol de decisión + entrenamiento**

In [100]:
tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_clf),
        ("model", DecisionTreeClassifier(
            max_depth=4,
            min_samples_leaf=50,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

tree_results = evaluate_classifier_cv(
    "Arbol de decision",
    tree_pipeline,
    X_clf,
    y_clf,
    clf_cv
)

print(f"ROC-AUC arbol de decision: {tree_results['roc_auc']:.3f}")
display(tree_results["confusion_matrix"])
display(tree_results["classification_report"].round(3))

ROC-AUC: 0.829


,Pred No,Pred Yes
Real No,3601,1562
Real Yes,339,1530


,precision,recall,f1-score,support
No,0.914,0.697,0.791,5163.0
Yes,0.495,0.819,0.617,1869.0


**5.3 Comparación con matriz de confusión y ROC**

In [101]:
comparison = pd.DataFrame([
    {
        "modelo": logistic_results["modelo"],
        "ROC-AUC": logistic_results["roc_auc"],
        "precision_Yes": logistic_results["classification_report"].loc["Yes", "precision"],
        "recall_Yes": logistic_results["classification_report"].loc["Yes", "recall"],
        "f1_Yes": logistic_results["classification_report"].loc["Yes", "f1-score"]
    },
    {
        "modelo": tree_results["modelo"],
        "ROC-AUC": tree_results["roc_auc"],
        "precision_Yes": tree_results["classification_report"].loc["Yes", "precision"],
        "recall_Yes": tree_results["classification_report"].loc["Yes", "recall"],
        "f1_Yes": tree_results["classification_report"].loc["Yes", "f1-score"]
    }
])

display(comparison.round(3))

roc_df = pd.concat([
    logistic_results["roc_curve"],
    tree_results["roc_curve"]
], ignore_index=True)

fig = px.line(
    roc_df,
    x="fpr",
    y="tpr",
    color="modelo",
    template=PLOTLY_TEMPLATE,
    title="Curva ROC: regresion logistica vs arbol de decision"
)
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1,
              line=dict(color="gray", dash="dash"))
fig.update_layout(
    font=PLOTLY_FONT,
    width=800,
    height=500,
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate"
)
fig.show()

,modelo,ROC-AUC,precision_Yes,recall_Yes,f1_Yes
0,Regresion logistica,0.845,0.516,0.799,0.627
1,Arbol de decision,0.829,0.495,0.819,0.617


**Discusión**

1. ¿Cuál modelo ganó? ¿En qué métrica?

   

2. ¿Qué cuesta más para el negocio: un falso positivo o un falso negativo?

   
3. ¿Cómo cambia tu elección de umbral si los costos no son simétricos?



## 6. Insights (3-5 hallazgos)

## 7. AI_USAGE — declaración integrada en una celda markdown final